# 07 Feature and Hybrid Ablation Study

An ablation changes one input group at a time while keeping the LSTM size, training dates, validation dates, forecast horizon and test keys consistent. This lets us measure which components actually help.

## 1. Locate the project

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root located; Phase 2 modules are importable.")

Project root located; Phase 2 modules are importable.


## 2. Run six controlled experiments

The suite compares target-only DeepAR, calendar inputs, static categories, price and availability, causal Kalman states, and the complete hybrid. Every v2 configuration is retrained so an incompatible v1 checkpoint cannot be reused.

In [2]:
from src.phase2.ablation import run_ablation_suite

ablation_results = run_ablation_suite(PROJECT_ROOT)
columns = [
    "feature_configuration", "MAE", "RMSE", "WAPE_percent",
    "mean_pinball_loss", "interval_coverage_80_percent",
    "mean_interval_width", "training_seconds",
]
display(ablation_results[columns])

,feature_configuration,MAE,RMSE,WAPE_percent,mean_pinball_loss,interval_coverage_80_percent,mean_interval_width,training_seconds
0,deepar,0.922143,2.119425,60.454226,0.311310,93.892857,3.371429,6.200366
1,hybrid,0.955893,2.272801,62.666823,0.316069,93.250000,3.191893,5.538953
2,target_only,0.917500,2.164734,60.149848,0.318408,93.464286,3.546536,5.675694
3,calendar,0.946250,2.185770,62.034652,0.322554,93.321429,3.566071,5.812669
4,kalman_only_addition,1.000893,2.472907,65.616952,0.327836,93.571429,3.234893,5.471833
5,calendar_static,1.105179,3.224668,72.453758,0.367148,93.107143,3.473536,6.167187


## 3. Interpret without cherry-picking

A component is considered helpful only when the controlled experiment improves relevant metrics. It is acceptable—and scientifically useful—for calendar or embedding features to fail on this small sample. The results table, not the intended story, determines the conclusion.

In [3]:
best = ablation_results.sort_values("mean_pinball_loss").iloc[0]
print("Best probabilistic configuration:", best["feature_configuration"])
print("Mean pinball loss:", round(best["mean_pinball_loss"], 4))

Best probabilistic configuration: deepar
Mean pinball loss: 0.3113
